# Graph Data Science with Neo4j & Python

**Author:** Matías Fernández Lakatos — *Neo4j Certified Professional*  
**GitHub:** [MFLakatos](https://github.com/MFLakatos)  
**Neo4j Profile:** [graphacademy.neo4j.com/u/34d98b08...](https://graphacademy.neo4j.com/u/34d98b08-78b3-4f35-a97c-3c869a7dec4e/)  
**Portfolio:** [mflakatos.github.io/MFernandezLakatos.github.io](https://mflakatos.github.io/MFernandezLakatos.github.io/)

---

## What this notebook covers

Graph databases model relationships as first-class citizens — something that traditional relational databases handle poorly. This notebook demonstrates:

1. **Setting up Neo4j** (local or AuraDB free tier)
2. **Loading a public dataset** (MovieLens 100K) into a property graph
3. **Cypher queries** for exploration and analytics
4. **Graph Data Science (GDS)** algorithms: PageRank, Community Detection (Louvain), Node Similarity
5. **Visualisation** of the graph structure

**Why graph for data science?**  
Graphs naturally represent recommendation systems, fraud detection networks, knowledge graphs, and social networks. Neo4j's GDS library gives you production-ready implementations of 60+ graph algorithms — directly callable from Python.

---

## 1. Setup

### Option A — Local Neo4j (recommended for exploration)
```bash
# Pull and run Neo4j with the GDS plugin
docker run \
  --name neo4j-gds \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  -e NEO4JLABS_PLUGINS='["graph-data-science"]' \
  neo4j:5
```

### Option B — Neo4j AuraDB Free Tier
Create a free instance at [console.neo4j.io](https://console.neo4j.io) and use the connection string provided.

In [ ]:
# Install dependencies
# !pip install neo4j pandas numpy matplotlib requests

from neo4j import GraphDatabase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests, io, zipfile, os
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# ── Connection ──────────────────────────────────────────────────
# Update these if using AuraDB
NEO4J_URI      = 'bolt://localhost:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'password'  # change to your password

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j!')


def run_query(cypher, params=None):
    """Helper: run a Cypher query and return results as a DataFrame."""
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return pd.DataFrame([r.data() for r in result])

## 2. Load MovieLens 100K Dataset

The [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) dataset contains 100,000 ratings from 943 users on 1,682 movies — a classic for recommendation systems and graph analytics.

In [ ]:
# Download MovieLens 100K
print('Downloading MovieLens 100K...')
url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall('.')

# Load ratings
ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['userId', 'movieId', 'rating', 'timestamp']
)

# Load movie metadata
movie_cols = ['movieId', 'title', 'release_date', 'video_release_date', 'imdb_url'] + \
             ['genre_' + g for g in ['unknown','action','adventure','animation',
              'children','comedy','crime','documentary','drama','fantasy',
              'film_noir','horror','musical','mystery','romance','sci_fi',
              'thriller','war','western']]
movies = pd.read_csv('ml-100k/u.item', sep='|', encoding='latin-1',
                     names=movie_cols)

print(f'Ratings: {len(ratings):,}')
print(f'Users:   {ratings.userId.nunique():,}')
print(f'Movies:  {ratings.movieId.nunique():,}')
ratings.head()

## 3. Build the Graph in Neo4j

**Graph model:**
```
(:User {userId}) -[:RATED {rating}]-> (:Movie {movieId, title, genres})
```

In [ ]:
# Clear existing data
run_query('MATCH (n) DETACH DELETE n')
print('Database cleared.')

# Create constraints (unique IDs)
run_query('CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.userId IS UNIQUE')
run_query('CREATE CONSTRAINT movie_id IF NOT EXISTS FOR (m:Movie) REQUIRE m.movieId IS UNIQUE')
print('Constraints created.')

In [ ]:
# Load Movies
genre_cols = [c for c in movies.columns if c.startswith('genre_')]

def get_genres(row):
    return [c.replace('genre_', '') for c in genre_cols if row[c] == 1]

movie_data = []
for _, row in movies.iterrows():
    movie_data.append({
        'movieId': int(row['movieId']),
        'title':   str(row['title']),
        'genres':  get_genres(row)
    })

with driver.session() as s:
    s.run("""
        UNWIND $movies AS m
        MERGE (movie:Movie {movieId: m.movieId})
        SET movie.title = m.title, movie.genres = m.genres
    """, movies=movie_data)
print(f'Loaded {len(movie_data)} movies.')

# Load Users
user_ids = [{'userId': int(u)} for u in ratings.userId.unique()]
with driver.session() as s:
    s.run("""
        UNWIND $users AS u
        MERGE (:User {userId: u.userId})
    """, users=user_ids)
print(f'Loaded {len(user_ids)} users.')

# Load Ratings (in batches of 5000)
batch_size = 5000
rating_records = ratings[['userId','movieId','rating']].to_dict('records')
for i in range(0, len(rating_records), batch_size):
    batch = rating_records[i:i+batch_size]
    batch_clean = [{'userId': int(r['userId']), 'movieId': int(r['movieId']), 'rating': float(r['rating'])} for r in batch]
    with driver.session() as s:
        s.run("""
            UNWIND $ratings AS r
            MATCH (u:User {userId: r.userId})
            MATCH (m:Movie {movieId: r.movieId})
            MERGE (u)-[rel:RATED]->(m)
            SET rel.rating = r.rating
        """, ratings=batch_clean)
print(f'Loaded {len(rating_records):,} ratings.')

## 4. Cypher Exploration Queries

In [ ]:
# Most-rated movies
df = run_query("""
    MATCH (u:User)-[r:RATED]->(m:Movie)
    RETURN m.title AS title,
           count(r) AS num_ratings,
           round(avg(r.rating), 2) AS avg_rating
    ORDER BY num_ratings DESC
    LIMIT 15
""")
print('Top 15 most-rated movies:')
print(df.to_string(index=False))

In [ ]:
# Users who rated the most movies (high-degree nodes)
df = run_query("""
    MATCH (u:User)-[r:RATED]->(m:Movie)
    RETURN u.userId AS userId,
           count(m) AS movies_rated,
           round(avg(r.rating), 2) AS avg_given
    ORDER BY movies_rated DESC
    LIMIT 10
""")
print('Top 10 most active users:')
print(df.to_string(index=False))

In [ ]:
# Users who share movies (2-hop path between users via common movies)
df = run_query("""
    MATCH (u1:User)-[:RATED]->(m:Movie)<-[:RATED]-(u2:User)
    WHERE u1.userId < u2.userId
    RETURN u1.userId AS user1, u2.userId AS user2,
           count(m) AS shared_movies
    ORDER BY shared_movies DESC
    LIMIT 10
""")
print('User pairs with most shared movies (basis for collaborative filtering):')
print(df.to_string(index=False))

## 5. Graph Data Science: PageRank

PageRank measures node **importance** in the graph. Applied to a movie-user graph, it identifies movies that are central to the rating network — not just popular, but connected to diverse, influential users.

In [ ]:
# Project a GDS graph (in-memory graph projection)
# We project a User-Movie bipartite graph weighted by rating
run_query("""
    CALL gds.graph.drop('movielens', false)
""")

run_query("""
    CALL gds.graph.project(
        'movielens',
        ['User', 'Movie'],
        {
            RATED: {
                orientation: 'UNDIRECTED',
                properties: 'rating'
            }
        }
    )
""")
print('GDS graph projected.')

In [ ]:
# Run PageRank and write scores back to the database
run_query("""
    CALL gds.pageRank.write('movielens', {
        maxIterations: 20,
        dampingFactor: 0.85,
        writeProperty: 'pagerank'
    })
    YIELD nodePropertiesWritten, ranIterations
""")
print('PageRank computed and written.')

# Retrieve top movies by PageRank
df_pr = run_query("""
    MATCH (m:Movie)
    WHERE m.pagerank IS NOT NULL
    RETURN m.title AS title, round(m.pagerank, 4) AS pagerank
    ORDER BY pagerank DESC
    LIMIT 15
""")
print('\nTop 15 movies by PageRank (network centrality):')
print(df_pr.to_string(index=False))

## 6. Community Detection: Louvain Algorithm

Louvain detects **communities** (clusters) of users who rate similar movies. This is directly useful for segmentation and collaborative filtering.

In [ ]:
run_query("""
    CALL gds.louvain.write('movielens', {
        writeProperty: 'community'
    })
    YIELD communityCount, modularity, modularities
""")
print('Louvain community detection complete.')

# Community size distribution
df_comm = run_query("""
    MATCH (n)
    WHERE n.community IS NOT NULL
    RETURN n.community AS community,
           count(*) AS size,
           labels(n)[0] AS node_type
    ORDER BY size DESC
    LIMIT 20
""")
print(df_comm.to_string(index=False))

In [ ]:
# What genres dominate each community?
df_genre_comm = run_query("""
    MATCH (u:User)-[:RATED]->(m:Movie)
    WHERE u.community IS NOT NULL AND size(m.genres) > 0
    WITH u.community AS community,
         [g IN m.genres WHERE g <> 'unknown' | g] AS genres
    UNWIND genres AS genre
    RETURN community, genre, count(*) AS freq
    ORDER BY community, freq DESC
""")

# Top genre per community
top_genre = df_genre_comm.groupby('community').first().reset_index()[['community','genre','freq']]
print('Dominant genre per user community:')
print(top_genre.head(10).to_string(index=False))

## 7. Node Similarity — Collaborative Filtering Foundation

In [ ]:
# Project a User-only graph based on shared movies (for similarity)
run_query("CALL gds.graph.drop('user-similarity', false)")

run_query("""
    CALL gds.graph.project.cypher(
        'user-similarity',
        'MATCH (u:User) RETURN id(u) AS id',
        'MATCH (u1:User)-[:RATED]->(m:Movie)<-[:RATED]-(u2:User)
         WHERE u1 <> u2
         RETURN id(u1) AS source, id(u2) AS target, count(m) AS weight'
    )
""")
print('User similarity graph projected.')

# Node Similarity (Jaccard)
df_sim = run_query("""
    CALL gds.nodeSimilarity.stream('user-similarity', {
        topK: 5,
        similarityCutoff: 0.1
    })
    YIELD node1, node2, similarity
    RETURN gds.util.asNode(node1).userId AS user1,
           gds.util.asNode(node2).userId AS user2,
           round(similarity, 4) AS jaccard_similarity
    ORDER BY jaccard_similarity DESC
    LIMIT 15
""")
print('Most similar user pairs (Jaccard):')
print(df_sim.to_string(index=False))

## 8. Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Plot 1: Genre distribution ──────────────────────────────
genre_counts = run_query("""
    MATCH (m:Movie)
    UNWIND m.genres AS genre
    WHERE genre <> 'unknown'
    RETURN genre, count(*) AS count
    ORDER BY count DESC
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genre bar chart
ax = axes[0]
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(genre_counts)))
ax.barh(genre_counts['genre'][::-1], genre_counts['count'][::-1], color=colors[::-1])
ax.set_title('Movie Count by Genre', fontweight='bold')
ax.set_xlabel('Number of Movies')
ax.grid(axis='x', alpha=0.3)

# ── Plot 2: Rating distribution ─────────────────────────────
ax2 = axes[1]
ratings_dist = run_query("""
    MATCH ()-[r:RATED]->()
    RETURN r.rating AS rating, count(*) AS count
    ORDER BY rating
""")
ax2.bar(ratings_dist['rating'], ratings_dist['count'],
        color=['#1e3a8a','#0f766e','#7c3aed','#d97706','#dc2626'])
ax2.set_title('Rating Distribution', fontweight='bold')
ax2.set_xlabel('Rating')
ax2.set_ylabel('Count')
ax2.set_xticks([1,2,3,4,5])
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('MovieLens 100K — Graph Exploration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('neo4j_exploration.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: PageRank top movies ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
top10 = df_pr.head(10)
bars = ax.barh(top10['title'][::-1], top10['pagerank'][::-1],
               color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, 10)))
ax.set_title('Top 10 Movies by PageRank (Network Centrality)', fontweight='bold', fontsize=12)
ax.set_xlabel('PageRank Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('neo4j_pagerank.png', dpi=150)
plt.show()

In [ ]:
# Close connection
driver.close()
print('Done! Connection closed.')

## 9. What's Next?

This notebook demonstrates the fundamentals. From here you can:

- **Build a recommendation engine** using Node Similarity scores as a collaborative filter
- **Add a Knowledge Graph layer** with movie metadata (directors, actors, descriptions) and run semantic similarity
- **Use GraphRAG** — combine Neo4j with an LLM to answer natural language questions over the graph
- **Fraud detection** — the same graph patterns (shared attributes, community membership) are directly applicable to detecting fraudulent networks

---

**Author:** Matías Fernández Lakatos  
[LinkedIn](https://www.linkedin.com/in/mflakatos) · [Neo4j Profile](https://graphacademy.neo4j.com/u/34d98b08-78b3-4f35-a97c-3c869a7dec4e/) · [Portfolio](https://mflakatos.github.io/MFernandezLakatos.github.io/)